In [1]:
# ============================================================================
# 01_data_preparation.ipynb
# Load and clean every study domain; build binary targets; apply a single
# fixed stratified train/test split; one-hot encode; and write the processed
# tables plus a feature registry that every downstream notebook reads.
#
# Six domains are processed through one domain-general routine:
#   credit_personal  - personal-loan default        (primary, high-stakes)
#   credit_corporate - corporate credit rating       (investment vs speculative)
#   credit_taiwan    - Taiwan credit-card default
#   credit_gmsc      - "Give Me Some Credit" delinquency
#   income_adult     - Adult census income (>50K)
#   churn            - Telco customer churn          (control)
#
# Engineered/duplicate add-on columns (VIP, *_STD, *_YEARS duplicates) are
# dropped; each dataset's native raw features, including demographic columns,
# are kept as-is per benchmark convention. Categoricals are one-hot encoded
# with the first level held out as reference.
# ============================================================================

# [Imports and global configuration]
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# Paths are resolved relative to the project root so the code can be published
# unchanged; no absolute path is ever printed (author anonymity).
ROOT = Path("..")
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
DATA_DIR = ROOT / "data"
FIG_DIR = ROOT / "results" / "figures"
TAB_DIR = ROOT / "results" / "tables"
for d in [PROC_DIR, FIG_DIR, TAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# [Plotting style: grayscale, publication settings, no captions]
sns.set_theme(style="whitegrid", context="paper")
sns.set_palette(sns.color_palette("gray", n_colors=8))
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600,
    "font.family": "sans-serif", "axes.grid": True,
    "grid.color": "0.85", "grid.linewidth": 0.5,
    "axes.edgecolor": "0.3", "axes.labelcolor": "0.15", "text.color": "0.15",
    "xtick.color": "0.3", "ytick.color": "0.3",
    "image.cmap": "gray", "savefig.bbox": "tight", "savefig.transparent": False,
})

def save_fig(fig, name):
    # Save a figure as PNG and PDF at 600 dpi, with no caption. Only the file
    # stem is reported, never a path.
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
    print(f"saved figure: {name}")

def save_table(df, name, index=False):
    df.to_csv(TAB_DIR / f"{name}.csv", index=index, encoding="utf-8-sig")
    print(f"saved table: {name}")

def sanitize(cols):
    # Make one-hot column names file- and model-safe (no spaces or symbols).
    out = []
    for c in cols:
        s = "".join(ch if (ch.isalnum() or ch == "_") else "_" for ch in str(c))
        while "__" in s:
            s = s.replace("__", "_")
        out.append(s.strip("_"))
    return out

# [Domain specifications]
# Each spec declares how to read the raw file, how to build the binary target,
# which columns to drop (ids, leakage-prone engineered flags, duplicates), and
# which raw columns are categorical. Everything else numeric is kept as-is.
INVESTMENT_GRADE = {"AAA", "AA", "A", "BBB"}
DOMAINS = {
    "credit_personal": dict(
        label="Personal-loan default", kind="credit", primary=True, control=False,
        file="credit_risk_dataset.csv", fmt="csv",
        target_col="loan_status", positive_label=1, positive_meaning="default",
        drop_cols=[],
        categorical=["person_home_ownership", "loan_intent", "loan_grade",
                     "cb_person_default_on_file"]),
    "credit_corporate": dict(
        label="Corporate credit rating", kind="credit", primary=False, control=False,
        file="corporate_rating.csv", fmt="csv",
        target_build="corp_invgrade", positive_meaning="investment grade",
        drop_cols=["Name", "Symbol", "Rating Agency Name", "Date", "Rating"],
        categorical=["Sector"]),
    "credit_taiwan": dict(
        label="Taiwan credit-card default", kind="credit", primary=False, control=False,
        file="taiwan_raw.parquet", fmt="parquet",
        target_col="DEFAULT", positive_label=1, positive_meaning="default",
        drop_cols=["VIP", "AGE_YEARS", "SEX_STD"],
        categorical=["EDUCATION", "MARRIAGE"]),
    "credit_gmsc": dict(
        label="Give Me Some Credit delinquency", kind="credit", primary=False, control=False,
        file="gmsc_raw.parquet", fmt="parquet",
        target_col="DEFAULT", positive_label=1, positive_meaning="serious delinquency",
        drop_cols=["VIP"],
        categorical=[]),
    "income_adult": dict(
        label="Adult census income", kind="income", primary=False, control=False,
        file="adult_raw.parquet", fmt="parquet",
        target_col="HIGH_INCOME", positive_label=1, positive_meaning="income above 50K",
        drop_cols=["VIP", "AGE_YEARS", "SEX_STD", "EDU_STD", "income", "fnlwgt"],
        categorical=["workclass", "education", "marital-status", "occupation",
                     "relationship", "race", "sex", "native-country"]),
    "churn": dict(
        label="Telco customer churn", kind="churn", primary=False, control=True,
        file="telco_churn.csv", fmt="csv",
        target_col="Churn", positive_label="Yes", positive_meaning="churn",
        drop_cols=["customerID"],
        categorical=None),  # auto-detect non-numeric columns
}

def load_raw(spec):
    p = RAW_DIR / spec["file"]
    return pd.read_csv(p) if spec["fmt"] == "csv" else pd.read_parquet(p)

def build_target(df, spec):
    # Return (dataframe without the target source columns, integer 0/1 target).
    if spec.get("target_build") == "corp_invgrade":
        y = df["Rating"].isin(INVESTMENT_GRADE).astype(int)
        return df, y
    col = spec["target_col"]
    if col == "Churn":
        y = (df[col].astype(str).str.strip() == "Yes").astype(int)
    else:
        y = (df[col] == spec["positive_label"]).astype(int)
    return df.drop(columns=[col]), y

def prepare(key, spec):
    df = load_raw(spec)
    # Telco TotalCharges arrives as text with blank strings for new accounts.
    if key == "churn":
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].replace(" ", np.nan),
                                           errors="coerce")
    df, y = build_target(df, spec)
    df = df.drop(columns=[c for c in spec["drop_cols"] if c in df.columns])
    # Treat the "?" placeholder in the Adult data as missing.
    df = df.replace("?", np.nan)
    # Drop exact duplicate rows (keep the aligned target).
    keep = ~df.duplicated()
    df, y = df.loc[keep].reset_index(drop=True), y.loc[keep].reset_index(drop=True)

    # Identify categorical columns. Non-numeric dtypes are treated as
    # categorical (robust across pandas object/string/str dtypes).
    if spec["categorical"] is None:
        cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    else:
        cat_cols = [c for c in spec["categorical"] if c in df.columns]
    num_cols = [c for c in df.columns if c not in cat_cols]

    # Impute: median for numeric, mode for categorical.
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())
    for c in cat_cols:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].mode(dropna=True).iloc[0])

    # One-hot encode categoricals, first level held out as the reference.
    X = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    X.columns = sanitize(X.columns)
    X = X.astype(float)
    features = list(X.columns)

    # Single stratified 70/30 split shared by every model and condition.
    idx = np.arange(len(X))
    tr, te = train_test_split(idx, test_size=0.30, random_state=SEED, stratify=y)
    reg = dict(
        label=spec["label"], kind=spec["kind"],
        primary=spec.get("primary", False), control=spec.get("control", False),
        target="target", positive_meaning=spec["positive_meaning"],
        features=features,
        numeric_original=[c for c in num_cols],
        categorical_original=[c for c in cat_cols],
        n_total=int(len(X)), n_train=int(len(tr)), n_test=int(len(te)),
        n_features=len(features), positive_rate=float(round(y.mean(), 4)),
    )
    # Persist processed splits with a stable per-domain case id.
    for part, ix in (("train", tr), ("test", te)):
        out = X.iloc[ix].copy()
        out.insert(0, "__case_id", [f"{key}_{i:06d}" for i in ix])
        out["target"] = y.iloc[ix].values
        out.to_parquet(PROC_DIR / f"{key}_{part}.parquet", index=False)
    return reg

# [Run preparation across all domains]
registry = {}
summary_rows = []
for key, spec in DOMAINS.items():
    reg = prepare(key, spec)
    registry[key] = reg
    summary_rows.append({
        "domain": key, "label": reg["label"], "role":
        "primary" if reg["primary"] else ("control" if reg["control"] else "added"),
        "n_total": reg["n_total"], "n_train": reg["n_train"], "n_test": reg["n_test"],
        "n_features": reg["n_features"], "positive_rate": reg["positive_rate"],
        "positive_meaning": reg["positive_meaning"],
    })
    print(f"{key:16s} n={reg['n_total']:>6d}  feat={reg['n_features']:>3d}  "
          f"pos_rate={reg['positive_rate']:.3f}  [{reg['label']}]")

with open(DATA_DIR / "feature_registry.json", "w", encoding="utf-8") as f:
    json.dump(registry, f, indent=1, ensure_ascii=False)
print("saved feature_registry.json")

summary = pd.DataFrame(summary_rows)
save_table(summary, "tab01_domain_summary")

# [Figure 01: class balance across domains]
order = list(DOMAINS.keys())
fig, ax = plt.subplots(figsize=(7.2, 3.6))
bars = ax.bar(range(len(order)), [registry[k]["positive_rate"] for k in order],
              color="0.45", edgecolor="0.15", width=0.62)
for i, k in enumerate(order):
    ax.text(i, registry[k]["positive_rate"] + 0.012,
            f"{registry[k]['positive_rate']:.2f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([registry[k]["label"] for k in order], rotation=28, ha="right", fontsize=8)
ax.set_ylabel("Positive-class rate")
ax.set_ylim(0, max(registry[k]["positive_rate"] for k in order) * 1.25)
save_fig(fig, "fig01_class_balance")

# [Figure 02: dataset size and dimensionality]
fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.4))
axes[0].bar(range(len(order)), [registry[k]["n_total"] for k in order],
            color="0.55", edgecolor="0.15", width=0.62)
axes[0].set_yscale("log"); axes[0].set_ylabel("Rows (log scale)")
axes[1].bar(range(len(order)), [registry[k]["n_features"] for k in order],
            color="0.35", edgecolor="0.15", width=0.62)
axes[1].set_ylabel("Features after encoding")
for ax in axes:
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=35, ha="right", fontsize=7)
save_fig(fig, "fig02_dataset_dimensions")

# [Figure 03: primary-domain numeric feature distributions by outcome]
prim = next(k for k, r in registry.items() if r["primary"])
tr = pd.read_parquet(PROC_DIR / f"{prim}_train.parquet")
num_feats = [c for c in registry[prim]["features"]
             if tr[c].nunique() > 8][:4]
fig, axes = plt.subplots(1, len(num_feats), figsize=(3.0 * len(num_feats), 3.0))
for ax, feat in zip(np.atleast_1d(axes), num_feats):
    for lab, shade in ((0, "0.65"), (1, "0.25")):
        vals = tr.loc[tr["target"] == lab, feat]
        vals = vals[(vals >= vals.quantile(0.01)) & (vals <= vals.quantile(0.99))]
        ax.hist(vals, bins=30, color=shade, alpha=0.75,
                label=("positive" if lab == 1 else "negative"), density=True)
    ax.set_xlabel(feat, fontsize=7); ax.set_yticks([])
    ax.legend(fontsize=6, frameon=False)
save_fig(fig, "fig03_primary_feature_distributions")

print("\nData preparation complete:",
      f"{len(registry)} domains, "
      f"{sum(r['n_total'] for r in registry.values()):,} total rows.")


credit_personal  n= 32416  feat= 22  pos_rate=0.219  [Personal-loan default]
credit_corporate n=  2029  feat= 36  pos_rate=0.574  [Corporate credit rating]


credit_taiwan    n= 29944  feat= 30  pos_rate=0.221  [Taiwan credit-card default]


credit_gmsc      n=149192  feat= 10  pos_rate=0.067  [Give Me Some Credit delinquency]


income_adult     n= 41439  feat= 96  pos_rate=0.239  [Adult census income]
churn            n=  7003  feat= 30  pos_rate=0.264  [Telco customer churn]
saved feature_registry.json
saved table: tab01_domain_summary


saved figure: fig01_class_balance


saved figure: fig02_dataset_dimensions


saved figure: fig03_primary_feature_distributions

Data preparation complete: 6 domains, 262,023 total rows.
